# Part 1 (실습) — 첫 LLM 호출 제대로 다루기

> **이 노트북의 목표**
> Part 0에서 "Hello, LangChain" 한 줄을 찍어봤다면, 이번엔 그 호출의 **속을 들여다봄**. 메시지 세 종류를 직접 조립하고, `temperature`로 답이 어떻게 달라지는지 눈으로 확인하고, 멀티턴 대화와 `.stream()`까지 손으로 돌려봄.
>
> **사용 모델**: `gemini-3-flash` (무료 등급)
> **선행**: Part 0 노트북 (설치·키 발급)

## 0. 준비 (Part 0 복습)

설치와 키 발급은 Part 0에서 끝냈음. 새 환경(코랩 새 세션 등)이라면 아래를 한 번 실행함.

In [ ]:
%pip install -q -U langchain langchain-google-genai

In [ ]:
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0.7)
print("준비 완료")

## 1. 메시지 세 종류 직접 조립하기

### 개념
ChatModel은 **역할(role)이 붙은 메시지**로 대화함.
- `SystemMessage` : 모델의 정체성·규칙 (무대 뒤 役 지시문)
- `HumanMessage` : 실제 질문
- `AIMessage` : 모델의 답

### 문법
`from langchain_core.messages import SystemMessage, HumanMessage`
각 메시지의 핵심 인수는 `content`(텍스트)임. 모델에는 **메시지들의 리스트**를 넘김.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="너는 리리마켓의 친절한 한국어 쇼핑 도우미야. 답은 항상 2문장 이내로 해."),
    HumanMessage(content="가을에 어울리는 니트 코디 추천해줘."),
]

res = llm.invoke(messages)
print(res.content)

### 직접 비교: SystemMessage가 답을 어떻게 바꾸는가
같은 질문에 役 지시문만 바꿔봄. 말투가 완전히 달라지는 것을 관찰해봄.

In [ ]:
roles = {
    "친절한 도우미": "너는 다정하고 친절한 쇼핑 도우미야.",
    "퉁명스러운 평론가": "너는 매우 퉁명스럽고 까칠한 패션 평론가야.",
}
question = HumanMessage(content="베이지 니트 어때?")

for name, system_prompt in roles.items():
    res = llm.invoke([SystemMessage(content=system_prompt), question])
    print(f"[{name}]")
    print(res.content)
    print("-" * 50)

## 2. temperature — 일관성 vs 다양성

### 개념
`temperature`는 답변의 무작위성을 조절하는 손잡이임.
- `0`에 가까움 → 일관·보수적 (같은 질문에 거의 같은 답). 추출·분류에 적합.
- `1`에 가까움 → 다양·창의적 (매번 다른 답). 카피라이팅·아이디어에 적합.

### 원리
LLM은 다음 단어를 확률 분포로 고름. temperature가 낮으면 가장 확률 높은 단어만, 높으면 덜 확률적인 단어도 선택됨.

### 실험
같은 슬로건 요청을 `temperature=0`과 `temperature=1`로 각각 3번씩 돌려봄.

In [ ]:
def make_slogans(temp, n=3):
    model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=temp)
    print(f"===== temperature = {temp} =====")
    for i in range(n):
        r = model.invoke("리리마켓 가을 세일 슬로건을 한 줄로 만들어줘. 슬로건만 출력해.")
        print(f"  {i+1}. {r.content.strip()}")
    print()

make_slogans(0.0)   # 거의 매번 비슷한 결과
make_slogans(1.0)   # 매번 다른 결과

> 📌 **관찰 포인트**: `temperature=0`은 3번 모두 비슷한 문구가, `temperature=1`은 매번 다른 문구가 나옴. "정답이 있는 작업이면 0, 다양성이 필요하면 높게"라는 감각을 여기서 체득함.

## 3. 멀티턴 대화 — 모델은 기억력이 없다

### 핵심 원리
> 모델은 매 호출이 독립적임. 맥락이 이어지는 건 **우리가 이전 대화를 리스트에 담아 다시 보내기 때문**임.

아래에서 모델의 답(`AIMessage`)까지 리스트에 추가해 "그거 말고"가 통하게 만들어봄.

In [ ]:
from langchain_core.messages import AIMessage

chat = [SystemMessage(content="너는 친절한 쇼핑 도우미야. 답은 짧게.")]

# 1턴
chat.append(HumanMessage(content="니트 하나 추천해줘."))
r1 = llm.invoke(chat)
print("AI:", r1.content)
chat.append(r1)   # ★ 모델의 답도 리스트에 추가해야 맥락이 이어짐

# 2턴 — 이전 맥락을 이어받는 질문
chat.append(HumanMessage(content="그거 말고 좀 더 캐주얼한 걸로."))
r2 = llm.invoke(chat)
print("AI:", r2.content)

print("\n현재 대화 리스트 길이:", len(chat), "개 메시지")

### 반례: 맥락을 안 넣으면 어떻게 되는가
마지막 질문만 단독으로 보내면 모델은 "그거"가 뭔지 모름.

In [ ]:
lonely = llm.invoke("그거 말고 좀 더 캐주얼한 걸로.")
print(lonely.content)
# → 무엇에 대한 것인지 몰라 엉뚱하거나 되묻는 답이 나옴

## 4. 응답 객체 해부 — AIMessage

`.invoke()`가 돌려주는 것은 문자열이 아니라 **`AIMessage` 객체**임. 본문 말고도 토큰 사용량 등 메타데이터가 들어 있음.

In [ ]:
res = llm.invoke("LangChain을 한 문장으로 설명해줘.")
print("타입:", type(res).__name__)
print("본문 (.content):", res.content)
print("토큰 사용량 (.usage_metadata):", res.usage_metadata)

## 5. 호출 메서드 — invoke / stream / batch

### 개념
- `.invoke(x)` : 한 번에 받기 (택배)
- `.stream(x)` : 실시간 조각으로 받기 (라이브 자막)
- `.batch([..])` : 여러 입력 한꺼번에 (공동구매)

### stream — 타이핑되듯 출력

In [ ]:
print("스트리밍 출력:\n")
for chunk in llm.stream("리리마켓을 소개하는 짧은 환영 인사를 써줘."):
    print(chunk.content, end="", flush=True)
print()

### batch — 여러 요청을 한꺼번에

In [ ]:
results = llm.batch([
    "고양이를 한 문장으로 소개해줘.",
    "강아지를 한 문장으로 소개해줘.",
    "앵무새를 한 문장으로 소개해줘.",
])
for r in results:
    print("•", r.content.strip())

## ✏️ 미니 실습

**과제**: 아래 조건으로 직접 호출해봄.
1. SystemMessage로 "너는 데이터 분석 강사이고, 항상 중학생도 이해할 비유를 들어 설명한다"를 설정
2. `temperature=0.3`인 모델을 새로 만들기
3. "오버피팅이 뭐야?"를 물어보기

In [ ]:
# TODO: 직접 작성
# teacher = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=...)
# msgs = [SystemMessage(content="..."), HumanMessage(content="...")]
# print(teacher.invoke(msgs).content)

<details>
<summary>👉 정답 코드 보기</summary>

```python
teacher = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0.3)
msgs = [
    SystemMessage(content="너는 데이터 분석 강사이고, 항상 중학생도 이해할 비유를 들어 설명한다."),
    HumanMessage(content="오버피팅이 뭐야?"),
]
print(teacher.invoke(msgs).content)
```
</details>

## 정리 — 무엇을 했는가

| 절 | 한 일 | 핵심 |
|---|---|---|
| 1 | 메시지 3종 조립 | `SystemMessage` / `HumanMessage` / `AIMessage` |
| 2 | temperature 실험 | 0=일관, 1=다양 |
| 3 | 멀티턴 대화 | 모델은 기억 없음 → 리스트로 맥락 전달 |
| 4 | 응답 해부 | `.content`, `.usage_metadata` |
| 5 | 호출 메서드 | `.invoke` / `.stream` / `.batch` |

### 3줄 요약
1. 대화는 역할이 붙은 메시지 리스트이며, 정체성은 SystemMessage가 정함.
2. temperature로 일관성↔다양성을 조절하고, 멀티턴은 리스트로 맥락을 직접 전달함.
3. 실행은 invoke·stream·batch 세 가지이며, 응답은 AIMessage 객체임.

### ▶ 다음 (Part 2)
매번 문장을 손으로 쓰는 대신, 빈칸이 있는 **프롬프트 템플릿(PromptTemplate)**으로 질문을 찍어내는 법을 배움. LCEL 파이프(|)로 가는 직전 단계임.